# Documentation for the Jupyter Notebook

This Jupyter Notebook processes APCEMM output files to extract and prepare data for training or validation of a machine learning model. The notebook reads netCDF files, extracts relevant data, processes it, and saves the processed data into a `.npy` file.

## Imports
This cell imports the necessary libraries for the notebook:
- `os.path` for file path manipulations.
- `xarray` for handling netCDF files.
- `numpy` for numerical operations.

## Pre-processing APCEMM outputs
Defines a class `apce_data_struct` to store the extracted data and a function `read_apcemm_data` to read and process APCEMM output files from a specified directory. The function extracts time, integrated optical depth (`int_OD`), and relative humidity (`RHi`) data from the netCDF files.

Initializes variables and sets parameters for processing the data:
- `training_sample_matrix_output` and `training_sample_matrix_input` to store the processed data.
- `scaled_mean` and `scaling_factor` for normalizing the data.
- `test`, `num_runs`, and `validation` to specify the test run number, number of runs, and whether the data is for validation or training.

## Save APCEMM output variables of interest
Loops through the specified number of runs, reads the APCEMM output data and corresponding relative humidity data, processes the data, and appends it to the respective matrices. The data is normalized and reshaped as required.

## Saving data
Saves the processed data (`training_sample_matrix`) into a `.npy` file for later use in training or validating the machine learning model.

In [1]:
#Import Libs
import os.path
import xarray as xr
import numpy as np

In [2]:
#Functions that will be used for postprocessing
class apce_data_struct:
    def __init__(self, t, ds_t, int_OD, RHi):
        self.t = t
        self.ds_t = ds_t
        self.int_OD = int_OD
        self.RHi = RHi
    
def read_apcemm_data(directory):
    t_mins = []
    ds_t = []
    int_OD = []
    RHi = []

    for file in sorted(os.listdir(directory)):
        if(file.startswith('ts_aerosol') and file.endswith('.nc')):
            file_path = os.path.join(directory,file)
            ds = xr.open_dataset(file_path, engine = "netcdf4", decode_times = False)
            ds_t.append(ds)
            tokens = file_path.split('.')
            mins = int(tokens[-2][-2:])
            hrs = int(tokens[-2][-4:-2])
            t_mins.append(hrs*60 + mins)
            int_OD.append(ds["intOD"])
            RHi.append(ds["RHi"])

    return apce_data_struct(t_mins, ds_t, int_OD, RHi)

In [3]:
# Extract integrated vertical optical depth data from APCEMM output files
training_sample_matrix_output = []
training_sample_matrix_input = []
training_sample_arrays = []
scaled_mean = 120 # Reducing back to [0,1] space after APCEMM scaling
scaling_factor = 15 # Reducing back to [0,1] space after APCEMM scaling
test = 1 # Test run number
num_runs = 2 # Number of test runs (either training or validation)
validation = True # Are you processing a validation set? If False assumes training set

if validation == True: # If you are processing a validation set
    run_type = "validation"
else: # If you are processing a training set
    run_type = "training"

In [ ]:
for i in range(1, num_runs+1): # For test runs num_runs
    if validation == True: # If you are processing a validation set
        apce_data = read_apcemm_data('/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/test_' + str(test) + '/outputs/test_' + str(test) + '_validation_' + str(i))
        input_RHi_ds = xr.open_dataset('/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/test_' + str(test) + '/APCEMM_met_validation_'+ str(i) +'.nc')
    else: # If you are processing a training set
        apce_data = read_apcemm_data('/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/test_' + str(test) + '/outputs/test_' + str(test) + '_run_' + str(i))
        input_RHi_ds = xr.open_dataset('/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/test_' + str(test) + '/APCEMM_met_run_'+ str(i) +'.nc')
    
    int_OD = apce_data.int_OD
    current_training_sample_output = np.array(int_OD).reshape(1, 73)[0]
    training_sample_matrix_output.append(current_training_sample_output)
    
    # Define your input_RHi array (length 24)
    input_RHi = input_RHi_ds['relative_humidity_ice'][98].values
    
    # Expand input_RHi to match the timestamps output by APCEMM
    expanded_input_RHi = [] 
    expanded_input_RHi.append(input_RHi[0]) # Count the zeroth timestep as a 7th repeat
    repeated_input_RHi = np.repeat(input_RHi[0:12], 6)
    expanded_input_RHi.extend(repeated_input_RHi) # 1 hour is just 6 10-minute intervals, and APCEMM is only run for 12 hours
    normalized_input_RHi = (np.array(expanded_input_RHi) - scaled_mean) / scaling_factor
    current_training_sample_input = np.array(normalized_input_RHi).reshape(1, 73)[0]
    training_sample_matrix_input.append(current_training_sample_input)
    
    # Stacking the input and output arrays
    stacked = np.dstack((current_training_sample_input, current_training_sample_output))
    training_sample_arrays.append(stacked)
    input_RHi_ds.close()

# This matrix will be used for training the machine learning model, it contains input RHi and output int_OD. In The future there will be multiple input variables.
training_sample_matrix = np.vstack(training_sample_arrays) # Shape: (10, 73, 2) INPUTS: (:,:,0), OUTPUTS: (:,:,1)

1
2


In [5]:
# Save the training_sample_matrix to a .npy file
np.save('/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/test_1/'+ run_type +'_sample_matrix.npy', training_sample_matrix)